In [1]:
import matminer
import pandas as pd
from matminer import featurizers
from matminer.datasets import load_dataset
from pymatgen.core.composition import Composition
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from matminer.featurizers.composition import ElementProperty, ElementFraction

In [2]:
# Superconductivity
df_sc = load_dataset("superconductivity2018")

In [3]:
df_sc

,composition,Tc
0,Ba0.4K0.6Fe2As2,31.20
1,Ca0.4Ba1.25La1.25Cu3O6.98,40.10
2,Mo0.39Ru0.61,6.90
3,Tm4Os6Sn19,1.10
4,Nd1Bi0.99Pb0.01S2F0.3O0.7,4.85
...,...,...
16409,Al4C3,0.00
16410,Nb0.96Ta0.04,8.87
16411,Pb2Sr2Ho0.5Ca0.5Cu2.982Al0.018O8,63.60
16412,Yb0.5Pr0.5Ba2Cu3O6.9,34.80


In [ ]:
# Define a function to create the target column
def is_semiconductor(Tc):
    # A semiconductor has a band gap between 0.1 and 4 eV
    return 0.1 <= Tc <= 4

In [ ]:
# Apply the function to the dataframe
df_sc['semiconductor'] = df_sc['Tc'].apply(is_semiconductor)

# Print the distribution of the target variable
print(df_sc['semiconductor'].value_counts())

In [4]:
composition = Composition("Ba0.4K0.6Fe2As2")
print(composition)


print(ef_features)

Ba0.4 K0.6 Fe2 As2
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.12, 0, 0, 0, 0, 0, 0, 0.4, 0, 0, 0, 0, 0, 0, 0.4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.08, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [5]:
# Split the dataframe into features and target
def create_composition(formula):
    try:
        return Composition(formula)
    except ValueError:
        print(f"Error parsing formula: {formula}")

df_sc["_Composition"] = df_sc['composition'].apply(create_composition).to_frame()
# y = df_sc['Tc']

Error parsing formula: Eu1.45Pr0.05Ce0.5Sr2Cu2Nb1O10=z
Error parsing formula: Sm1Ba-1Cu3O6.94
Error parsing formula: Y2C2Br0.5!1.5
Error parsing formula: Hg0.3Pb0.7Sr1.75La0.25Cu1O4+2
Error parsing formula: Hg1Sr2Ho0.333Ce0.667Cu2O6=z
Error parsing formula: B1Sr2Ca3Cu4O2N+3
Error parsing formula: B1Sr2Ca4Cu5O2N+3
Error parsing formula: B1Sr2Ca2Cu3O2N+3


In [6]:
df_sc

,composition,Tc,_Composition
0,Ba0.4K0.6Fe2As2,31.20,"(Ba, K, Fe, As)"
1,Ca0.4Ba1.25La1.25Cu3O6.98,40.10,"(Ca, Ba, La, Cu, O)"
2,Mo0.39Ru0.61,6.90,"(Mo, Ru)"
3,Tm4Os6Sn19,1.10,"(Tm, Os, Sn)"
4,Nd1Bi0.99Pb0.01S2F0.3O0.7,4.85,"(Nd, Bi, Pb, S, F, O)"
...,...,...,...
16409,Al4C3,0.00,"(Al, C)"
16410,Nb0.96Ta0.04,8.87,"(Nb, Ta)"
16411,Pb2Sr2Ho0.5Ca0.5Cu2.982Al0.018O8,63.60,"(Pb, Sr, Ho, Ca, Cu, Al, O)"
16412,Yb0.5Pr0.5Ba2Cu3O6.9,34.80,"(Yb, Pr, Ba, Cu, O)"


In [7]:
featurizer = ElementProperty.from_preset('magpie')
df_sc_ftd = featurizer.featurize_dataframe(df_sc, col_id='_Composition', ignore_errors=True)

ElementProperty:   0%|          | 0/16414 [00:00<?, ?it/s]

In [8]:
df_sc_ftd

,composition,Tc,_Composition,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,...,MagpieData range GSmagmom,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData mode GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber
0,Ba0.4K0.6Fe2As2,31.20,"(Ba, K, Fe, As)",19.0,56.0,37.0,30.360000,6.214400,26.0,3.0,...,2.110663,0.844265,1.013118,0.0,166.0,229.0,63.0,203.800000,30.240000,166.0
1,Ca0.4Ba1.25La1.25Cu3O6.98,40.10,"(Ca, Ba, La, Cu, O)",8.0,57.0,49.0,22.677795,16.074864,8.0,7.0,...,0.000000,0.000000,0.000000,0.0,12.0,229.0,217.0,106.949534,102.911141,12.0
2,Mo0.39Ru0.61,6.90,"(Mo, Ru)",42.0,44.0,2.0,43.220000,0.951600,44.0,50.0,...,0.000000,0.000000,0.000000,0.0,194.0,229.0,35.0,207.650000,16.653000,194.0
3,Tm4Os6Sn19,1.10,"(Tm, Os, Sn)",50.0,76.0,26.0,58.000000,10.482759,50.0,37.0,...,0.000000,0.000000,0.000000,0.0,141.0,194.0,53.0,159.275862,23.947681,141.0
4,Nd1Bi0.99Pb0.01S2F0.3O0.7,4.85,"(Nd, Bi, Pb, S, F, O)",8.0,83.0,75.0,36.658000,27.869600,16.0,19.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,72.206000,49.328776,70.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16409,Al4C3,0.00,"(Al, C)",6.0,13.0,7.0,10.000000,3.428571,13.0,73.0,...,0.000000,0.000000,0.000000,0.0,194.0,225.0,31.0,211.714286,15.183673,225.0
16410,Nb0.96Ta0.04,8.87,"(Nb, Ta)",41.0,73.0,32.0,42.280000,2.457600,41.0,47.0,...,0.000000,0.000000,0.000000,0.0,229.0,229.0,0.0,229.000000,0.000000,229.0
16411,Pb2Sr2Ho0.5Ca0.5Cu2.982Al0.018O8,63.60,"(Pb, Sr, Ho, Ca, Cu, Al, O)",8.0,82.0,74.0,27.138250,19.616202,8.0,7.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,117.531250,105.531250,12.0
16412,Yb0.5Pr0.5Ba2Cu3O6.9,34.80,"(Yb, Pr, Ba, Cu, O)",8.0,70.0,62.0,24.705426,17.870921,8.0,9.0,...,0.000000,0.000000,0.000000,0.0,12.0,229.0,217.0,110.488372,105.359654,12.0


In [10]:
df_sc_ftd.to_csv("df_sc_ep_ElementProperty_Magpie_ftd.csv", index = False)